<a href="https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(11694072, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [3]:
df_real = df[df['gsc_data_available'] == True].copy()
print(f"Rows before filtering: {len(df)}")
print(f"Rows after filtering (real GSC data only): {len(df_real)}")

Rows before filtering: 11694072
Rows after filtering (real GSC data only): 3878937


In [4]:
page = df_real.groupby(['client_hash_id', 'content_hash_id']).agg(
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    sum_position=('gsc_sum_position', 'sum')
).reset_index()

print(f"Number of unique pages: {len(page)}")
page.head()

Number of unique pages: 208636


,client_hash_id,content_hash_id,total_impressions,total_clicks,sum_position
0,client_06d356715a8ff3b6,content_0058bd88fb1821f2,241,1,2563
1,client_06d356715a8ff3b6,content_0059a4d4195810c9,845,2,10331
2,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,957,3,7633
3,client_06d356715a8ff3b6,content_0094c7d0fbcc07b7,147,0,4241
4,client_06d356715a8ff3b6,content_00a34394d4ee05ce,142,1,2606


In [5]:
page['avg_position'] = page['sum_position'] / page['total_impressions']
page['ctr'] = page['total_clicks'] / page['total_impressions']

# drop rows with zero impressions, they can't have a real position or CTR
page = page[page['total_impressions'] > 0].copy()

print(f"Pages remaining after dropping zero-impression rows: {len(page)}")
page[['total_impressions', 'total_clicks', 'avg_position', 'ctr']].head()

Pages remaining after dropping zero-impression rows: 208636


,total_impressions,total_clicks,avg_position,ctr
0,241,1,10.634855,0.004149
1,845,2,12.226036,0.002367
2,957,3,7.975967,0.003135
3,147,0,28.85034,0.000000
4,142,1,18.352113,0.007042


In [6]:
def position_bucket(p):
    if p <= 3:
        return '1_top3'
    elif p <= 10:
        return '2_top10'
    elif p <= 20:
        return '3_pos11to20'
    elif p <= 50:
        return '4_pos21to50'
    else:
        return '5_beyond50'

page['position_bucket'] = page['avg_position'].apply(position_bucket)

signal1 = page.groupby('position_bucket').agg(
    n=('ctr', 'size'),
    avg_ctr=('ctr', 'mean')
).round(4)

print(signal1)

                     n  avg_ctr
position_bucket                
1_top3           14044   0.0076
2_top10          77995   0.0058
3_pos11to20      41151   0.0046
4_pos21to50      44306   0.0030
5_beyond50       31140   0.0189


In [7]:
print(page[page['position_bucket'] == '5_beyond50']['total_impressions'].describe())

count     31140.000000
mean        102.450835
std         834.627133
min           1.000000
25%          14.000000
50%          22.000000
75%          87.000000
max      123819.000000
Name: total_impressions, dtype: float64


In [8]:
signal2 = page.groupby('position_bucket').agg(
    n=('total_impressions', 'size'),
    total_impressions=('total_impressions', 'sum')
)
signal2['share_of_all_impressions'] = (signal2['total_impressions'] / signal2['total_impressions'].sum()).round(4)

print(signal2)

                     n  total_impressions  share_of_all_impressions
position_bucket                                                    
1_top3           14044            9198602                    0.0425
2_top10          77995          157261774                    0.7274
3_pos11to20      41151           27482702                    0.1271
4_pos21to50      44306           19061475                    0.0882
5_beyond50       31140            3190319                    0.0148


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



Every page gets checked in this order. First match wins.

1. **Quick Win** — page ranks in positions 11 to 20 and has real impression volume.
2. **CTR Fix** — page ranks in the top 10 but its CTR is below the median CTR for its own position bucket.
3. **Refresh** — page is losing impressions over the month and still has real volume.
4. **Monitor** — none of the above. No action this cycle.

The score for any flagged page is impressions multiplied by the size of its performance gap. A page with more traffic riding on the same gap always outranks a smaller page with the same gap.

**Reason codes:** `QUICK_WIN`, `CTR_FIX`, `REFRESH`, `MONITOR`

---

### Signal 1: CTR vs. position (behind the CTR-fix flag)

| bucket | n | avg_ctr |
|---|---|---|
| top3 | 14,044 | 0.0076 |
| top10 | 77,995 | 0.0058 |
| pos11to20 | 41,151 | 0.0046 |
| pos21to50 | 44,306 | 0.0030 |
| beyond50 | 31,140 | 0.0189 |

CTR drops cleanly from top3 through pos21to50, matching the CTR-fix logic exactly. The beyond50 bucket breaks the pattern with a high average CTR, but checking its impression volume showed a median of only 22 impressions per page, so a handful of high-CTR outliers (likely branded searches) pull that bucket's average up. Not reliable evidence either way.

**Verdict: MIXED.** Confirmed for buckets 1 through 4. Bucket 5 is unreliable due to low sample volume, not a genuine reversal.

---

### Signal 2: volume by position (behind the quick-win flag)

| bucket | n | total_impressions | share_of_all |
|---|---|---|---|
| top3 | 14,044 | 9,198,602 | 4.25% |
| top10 | 77,995 | 157,261,774 | 72.74% |
| pos11to20 | 41,151 | 27,482,702 | 12.71% |
| pos21to50 | 44,306 | 19,061,475 | 8.82% |
| beyond50 | 31,140 | 3,190,319 | 1.48% |

Pages in position 11-20 hold 12.71% of total impressions across 41,151 pages, over 27 million impressions. That is a real, meaningful slice of traffic sitting just off page one, not a marginal group.

**Verdict: CONFIRMED.** Quick-win logic has real volume to act on.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# Volume threshold: don't chase pages nobody searches for
VOLUME_MIN = page['total_impressions'].median()
print(f"Volume threshold (median impressions): {VOLUME_MIN}")

# Median CTR per bucket, used to judge if a top-10 page underperforms its own bucket
page['bucket_median_ctr'] = page.groupby('position_bucket')['ctr'].transform('median')
print(page[['position_bucket', 'bucket_median_ctr']].drop_duplicates())

Volume threshold (median impressions): 91.0
    position_bucket  bucket_median_ctr
0       3_pos11to20           0.000000
2           2_top10           0.000924
3       4_pos21to50           0.000000
86       5_beyond50           0.000000
250          1_top3           0.000000


In [10]:
page['bucket_mean_ctr'] = page.groupby('position_bucket')['ctr'].transform('mean')
print(page[['position_bucket', 'bucket_mean_ctr']].drop_duplicates())

    position_bucket  bucket_mean_ctr
0       3_pos11to20         0.004595
2           2_top10         0.005811
3       4_pos21to50         0.002985
86       5_beyond50         0.018880
250          1_top3         0.007580


In [11]:
df_real['day'] = pd.to_datetime(df_real['report_date']).dt.day

first_half = df_real[df_real['day'] <= 15].groupby(['client_hash_id', 'content_hash_id'])['gsc_impressions'].sum()
second_half = df_real[df_real['day'] >= 16].groupby(['client_hash_id', 'content_hash_id'])['gsc_impressions'].sum()

page = page.set_index(['client_hash_id', 'content_hash_id'])
page['impr_first_half'] = first_half
page['impr_second_half'] = second_half
page = page.reset_index()

page['impr_first_half'] = page['impr_first_half'].fillna(0)
page['impr_second_half'] = page['impr_second_half'].fillna(0)

page['trend_pct'] = (page['impr_second_half'] - page['impr_first_half']) / page['impr_first_half'].replace(0, pd.NA)

print(page[['total_impressions', 'impr_first_half', 'impr_second_half', 'trend_pct']].head(10))

   total_impressions  impr_first_half  impr_second_half trend_pct
0                241             58.0             183.0  2.155172
1                845            688.0             157.0 -0.771802
2                957            572.0             385.0 -0.326923
3                147             95.0              52.0 -0.452632
4                142             43.0              99.0  1.302326
5                801             92.0             709.0  6.706522
6                170              0.0             170.0      <NA>
7                695            562.0             133.0 -0.763345
8                517            216.0             301.0  0.393519
9                797            481.0             316.0 -0.343035


In [12]:
def assign_reason(row):
    if row['position_bucket'] == '3_pos11to20' and row['total_impressions'] >= VOLUME_MIN:
        return 'QUICK_WIN'
    elif row['position_bucket'] in ['1_top3', '2_top10'] and row['ctr'] < row['bucket_mean_ctr'] and row['total_impressions'] >= VOLUME_MIN:
        return 'CTR_FIX'
    elif pd.notna(row['trend_pct']) and row['trend_pct'] <= -0.10 and row['total_impressions'] >= VOLUME_MIN:
        return 'REFRESH'
    else:
        return 'MONITOR'

page['reason_code'] = page.apply(assign_reason, axis=1)

action_map = {
    'QUICK_WIN': 'Push toward page 1',
    'CTR_FIX': 'Rewrite title/meta for CTR',
    'REFRESH': 'Refresh or expand content',
    'MONITOR': 'No action this cycle'
}
page['action'] = page['reason_code'].map(action_map)

print(page['reason_code'].value_counts())

reason_code
MONITOR      127198
CTR_FIX       32722
QUICK_WIN     26445
REFRESH       22271
Name: count, dtype: int64


In [15]:
target_ctr = page[page['position_bucket'] == '1_top3']['ctr'].mean()

def compute_score(row):
    if row['reason_code'] in ('QUICK_WIN', 'CTR_FIX'):
        gap = max(0, target_ctr - row['ctr'])
        return row['total_impressions'] * gap
    elif row['reason_code'] == 'REFRESH':
        return row['total_impressions'] * abs(row['trend_pct'])
    else:
        return 0.0

page['action_score'] = page.apply(compute_score, axis=1)

queue = page.sort_values('action_score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written: {len(queue)} rows")
queue[['client_hash_id', 'content_hash_id', 'reason_code', 'action', 'action_score']].head(10)

Queue written: 208636 rows


,client_hash_id,content_hash_id,reason_code,action,action_score
0,client_9c26c096d6e57253,content_adcc7b85a04c187d,REFRESH,Refresh or expand content,130340.770683
1,client_9c26c096d6e57253,content_54f2b96801c90591,REFRESH,Refresh or expand content,108635.832223
2,client_23a62021009f63c4,content_661a7734f691bef5,REFRESH,Refresh or expand content,63654.507544
3,client_23a62021009f63c4,content_c60628276389acbb,REFRESH,Refresh or expand content,57103.788912
4,client_86ebc2f12c01f586,content_14824df843e76fa8,REFRESH,Refresh or expand content,49290.208058
5,client_23a62021009f63c4,content_93a9b8328d4fd032,REFRESH,Refresh or expand content,44228.485550
6,client_0fa64a184f18a4a0,content_2db2a9dcb3b62a3a,REFRESH,Refresh or expand content,32102.325210
7,client_23a62021009f63c4,content_36e53e9c707674fc,REFRESH,Refresh or expand content,32070.705235
8,client_23a62021009f63c4,content_895766c2a54d3fd0,REFRESH,Refresh or expand content,29175.378728
9,client_8ddc46da5414ffd8,content_6bc3589504bab587,REFRESH,Refresh or expand content,27714.944104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top10 = queue.head(10)
print(top10[['reason_code', 'total_impressions', 'avg_position', 'ctr', 'trend_pct', 'action_score']].to_string())

  reason_code  total_impressions  avg_position       ctr trend_pct   action_score
0     REFRESH             283505      1.713769  0.536745 -0.459748  130340.770683
1     REFRESH             267068      1.723247  0.533991 -0.406772  108635.832223
2     REFRESH             106836     22.825293  0.001685 -0.595815   63654.507544
3     REFRESH             123819     88.483076  0.000048 -0.461188   57103.788912
4     REFRESH              66244      3.671548  0.013737 -0.744071   49290.208058
5     REFRESH              78070     22.538875  0.000986 -0.566523   44228.485550
6     REFRESH             144946      4.218702  0.009079 -0.221478   32102.325210
7     REFRESH              80453     26.518203  0.002648 -0.398627   32070.705235
8     REFRESH              90661      6.766316  0.009045 -0.321807   29175.378728
9     REFRESH              44988      2.451765  0.008536 -0.616052   27714.944104


### Top-10 review

1. **Refresh or expand content** — position 1.7, huge volume (283,505 impressions), impressions dropped 46% from first to second half of the month. Would be wrong if this is a seasonal dip that recovers next month, not a real decline.

2. **Refresh or expand content** — position 1.7, 267,068 impressions, down 41%. Same client as row 1, so this could be one site-wide issue (e.g. a technical problem or algorithm update) rather than two separate content problems.

3. **Refresh or expand content** — position 22.8, CTR near zero (0.0017), down 60%. Would be wrong if this page never had real traffic to lose, low CTR at position 22 is expected regardless of any decline.

4. **Refresh or expand content** — position 88.5, CTR essentially zero (0.00005), down 46%. Would be wrong for the same reason as row 3, a page ranking this low has little real traffic at stake, the score is inflated by trend_pct on a small base.

5. **Refresh or expand content** — position 3.7, 66,244 impressions, down 74%, the steepest drop in the top 10. Would be wrong if this was a one-time event (page temporarily de-indexed, site outage) rather than an ongoing decline.

6. **Refresh or expand content** — position 22.5, CTR near zero, down 57%. Same pattern as rows 3 and 4, low CTR at this position is normal, so the "decline" may not represent a meaningful loss.

7. **Refresh or expand content** — position 4.2, 144,946 impressions, down 22%, the mildest decline in the top 10. Would be wrong if 22% is within normal month-to-month variance for a page this size, not a genuine downward trend.

8. **Refresh or expand content** — position 26.5, CTR near zero, down 40%. Same low-CTR-at-low-position caveat as rows 3, 4, and 6.

9. **Refresh or expand content** — position 6.8, 90,661 impressions, down 32%. Would be wrong if a competitor's page temporarily outranked this one and normal position resumes without any content changes.

10. **Refresh or expand content** — position 2.5, 44,988 impressions, down 62%. Would be wrong if this is measurement noise from a smaller impression base swinging more per click.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
weak_picks = queue.head(10)[queue.head(10)['avg_position'] > 15]
print(weak_picks[['reason_code', 'total_impressions', 'avg_position', 'ctr', 'trend_pct']])

  reason_code  total_impressions  avg_position       ctr trend_pct
2     REFRESH             106836     22.825293  0.001685 -0.595815
3     REFRESH             123819     88.483076  0.000048 -0.461188
5     REFRESH              78070     22.538875  0.000986 -0.566523
7     REFRESH              80453     26.518203  0.002648 -0.398627


### Weak picks

Rows 3, 4, 6, and 8 in the top 10 all rank beyond position 20, with CTR near zero (0.0005 to 0.003). Their high action_score comes entirely from a large trend_pct multiplied by impressions, not from any real traffic at stake. A page with 100 impressions and 0 clicks that drops to 40 impressions still shows a large negative trend_pct, even though the actual loss is a handful of clicks. These four picks are likely false positives for the REFRESH label, they are mathematically "declining" but not meaningfully important to review first.

This is a real weakness in the current rule: REFRESH's score formula (impressions × trend_pct) can output very large numbers for low-CTR pages with big percentage swings on a small base, while QUICK_WIN and CTR_FIX scores are capped by a small CTR gap. The three reason codes are not on a comparable scale. A fix for Week 5: weight trend_pct by a page's absolute impression change (not just percent), or normalize each reason code's score before combining them into one ranked queue.

In [17]:
print("Leakage check:")
print(f"- trend_pct compares day 1-15 vs day 16-31 of the SAME month already in hand, no future month or future data used.")
print(f"- No FlyRank product flag columns exist in this table (checked df.columns), so no product decision flags were used as inputs.")
print(f"- bucket_mean_ctr and target_ctr are computed from this same sample month, not from a separate labeled outcome or held-out period.")
print(f"- VOLUME_MIN ({VOLUME_MIN}) is the median of the same data being scored, a simple choice, not tuned or leaked from anywhere else.")

Leakage check:
- trend_pct compares day 1-15 vs day 16-31 of the SAME month already in hand, no future month or future data used.
- No FlyRank product flag columns exist in this table (checked df.columns), so no product decision flags were used as inputs.
- bucket_mean_ctr and target_ctr are computed from this same sample month, not from a separate labeled outcome or held-out period.
- VOLUME_MIN (91.0) is the median of the same data being scored, a simple choice, not tuned or leaked from anywhere else.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.